# PSP Provider Pipeline — Production Run

Повний цикл: завантаження → препроцесинг → класифікація → judge → експорт.

**Pipeline per message:**
```
Raw file → reply merge → filter → dedup → pre_filter → stage1 LLM → full LLM → judge → Excel
```

In [1]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
INPUT_FILE   = "data/input/messages_corpus.csv"    # сирий файл з Telegram-експорту
SHEET        = None               # None = перший лист
OUTPUT_FILE  = "data/output/providers_production_v2.xlsx"
GLOSSARY_DB  = "data/provider_glossary.db"

MIN_TEXT_LEN    = 20
REQUEST_DELAY   = 0.1

# Вимкни щоб прискорити / здешевити прогін
PRE_FILTER_ENABLED = True
STAGE1_ENABLED     = True
JUDGE_ENABLED      = True
RAG_ENABLED  = True
RAG_DB_PATH  = "data/rag/chroma"

# Опціонально: окрема сильніша модель для judge
# Якщо None — використовує ту саму модель що й екстрактор
JUDGE_MODEL = None  # наприклад: 'openai/gpt-4.1-mini'

# ── IMPORTS ──────────────────────────────────────────────────────────────────
import os, sys, time, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('../.env'))
if JUDGE_MODEL:
    os.environ['PROVIDER_JUDGE_MODEL'] = JUDGE_MODEL

PROJECT_ROOT = str(Path('..').resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from glossary_builder.llm import LLMClient
from PSP_providers_glossary_builder.provider_classifier import (
    classify_provider_message,
    load_provider_glossary,
    make_provider_llm_client,
)
from PSP_providers_glossary_builder.provider_decision import ProviderExtractionConfig

print('OK')

OK


In [2]:
# ── ЗАВАНТАЖЕННЯ ─────────────────────────────────────────────────────────────
if str(INPUT_FILE).endswith('.csv'):
    raw = pd.read_csv(INPUT_FILE)
else:
    raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET, dtype={
        'message_id':      'Int64',
        'reply_to_msg_id': 'Int64',
        'user_id':         'Int64',
    })

raw['text']     = raw['text'].fillna('').astype(str).str.strip()
raw['username'] = raw.get('username', pd.Series([''] * len(raw))).fillna('').astype(str)

print(f'Завантажено:       {len(raw):>8,} рядків')
print(f'З текстом:         {(raw.text != "").sum():>8,}')
print(f'Колонки: {list(raw.columns)}')

Завантажено:         89,651 рядків
З текстом:           84,418
Колонки: ['id', 'message_id', 'text', 'date', 'group_id', 'user_id', 'username']


In [3]:
# ── ПРЕПРОЦЕСИНГ ─────────────────────────────────────────────────────────────
# Крок 0: склейка reply-chain (1 рівень)
msg_lookup = (
    raw.dropna(subset=['message_id'])
       .set_index('message_id')['text']
       .to_dict()
)

def merge_context(row):
    original = row['text']
    reply_id = row.get('reply_to_msg_id')
    if pd.notna(reply_id):
        ctx = msg_lookup.get(int(reply_id), '').strip()
        if ctx:
            return f'[context] {ctx}\n{original}'
    return original

df = raw.copy()
df['text_merged'] = df.apply(merge_context, axis=1)

n_ctx = df['text_merged'].str.startswith('[context]').sum()
print(f'Крок 0 — reply merge:   {n_ctx:,} повідомлень отримали контекст')

# Крок 1: фільтр коротких (по оригінальному тексту)
before = len(df)
df = df[df['text'].str.len() >= MIN_TEXT_LEN].reset_index(drop=True)
print(f'Крок 1 — фільтр <{MIN_TEXT_LEN}: {before - len(df):,} видалено | залишилось {len(df):,}')

# ── Крок 2+3: дедупликація (симуляція production deduplication.py) ────────────
import hashlib
from datetime import timedelta

HOLD_WINDOW = timedelta(minutes=3)   # same user, any text
TEXT_WINDOW  = timedelta(hours=24)   # same user, same text

def _hash(text: str) -> str:
    return hashlib.md5(text.strip().lower().encode()).hexdigest()

# Сортуємо по даті щоб симулювати послідовність як у вебхуці
if 'date' in df.columns:
    df['_date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.sort_values('_date_parsed').reset_index(drop=True)

# Стан дедупликатора: username → {text_hash, created_at}
dedup_state: dict[str, dict] = {}
keep_mask = []

for _, row in df.iterrows():
    username = str(row.get('user_id', '')).strip()
    text     = str(row.get('text', '')).strip()
    ts       = row.get('_date_parsed', pd.NaT)

    # Без юзернейму або без тексту — пропускаємо дедуп
    if not username or not text:
        keep_mask.append(True)
        continue

    new_hash = _hash(text)
    now      = ts if pd.notna(ts) else pd.Timestamp.now(tz='UTC')

    prev = dedup_state.get(username)

    if prev is None:
        # Перше повідомлення від цього юзера
        dedup_state[username] = {'text_hash': new_hash, 'created_at': now}
        keep_mask.append(True)
        continue

    diff = now - prev['created_at'] if pd.notna(prev['created_at']) and pd.notna(now) else timedelta(days=999)

    # Check 1: time-based (будь-яке повідомлення від того ж юзера < 3 хв)
    if diff < HOLD_WINDOW:
        keep_mask.append(False)
        continue

    # Check 2: text-based (той самий текст від того ж юзера < 24 год)
    if prev['text_hash'] == new_hash and diff < TEXT_WINDOW:
        keep_mask.append(False)
        continue

    # Не дублікат — оновлюємо стан
    dedup_state[username] = {'text_hash': new_hash, 'created_at': now}
    keep_mask.append(True)

before = len(df)
df = df[keep_mask].reset_index(drop=True)
print(f'Крок 2 — dedup (production sim): {before - len(df):,} дублів | залишилось {len(df):,}')

# Прибираємо допоміжну колонку
if '_date_parsed' in df.columns:
    df = df.drop(columns=['_date_parsed'])

Крок 0 — reply merge:   0 повідомлень отримали контекст
Крок 1 — фільтр <20: 7,404 видалено | залишилось 82,247
Крок 2 — dedup (production sim): 62,886 дублів | залишилось 19,361


In [4]:
# ── ІНІЦІАЛІЗАЦІЯ ─────────────────────────────────────────────────────────────
glossary = load_provider_glossary(GLOSSARY_DB)
print(f'Глосарій:  {len(glossary)} термінів')

# Використовуємо PSP_PROVIDERS_OPENROUTER_API_KEY якщо є, інакше fallback
try:
    llm = make_provider_llm_client()
    print(f'LLM:       OpenRouter / {llm.model}')
except ValueError:
    llm = LLMClient()
    print(f'LLM:       {llm.provider} / {llm.model}  (fallback)')

cfg = ProviderExtractionConfig(
    pre_filter_db_path = GLOSSARY_DB,
    pre_filter_enabled = PRE_FILTER_ENABLED,
    stage1_enabled     = STAGE1_ENABLED,
    judge_enabled      = JUDGE_ENABLED,
)
print(f'Config:    pre_filter={PRE_FILTER_ENABLED}  stage1={STAGE1_ENABLED}  judge={JUDGE_ENABLED}')

from glossary_builder.rag import RagIndex, RagConfig
provider_rag = None
if RAG_ENABLED:
    rag_cfg = RagConfig(
        db_path=RAG_DB_PATH,
        collection_name="provider_examples",
        api_key=os.environ.get("PSP_PROVIDERS_OPENROUTER_API_KEY", ""),
    )
    provider_rag = RagIndex(rag_cfg)
    print(f'RAG:       {provider_rag.count()} прикладів з {RAG_DB_PATH}')
else:
    print('RAG:       вимкнено')

Глосарій:  331 термінів
LLM:       OpenRouter / openai/gpt-4.1-nano
Config:    pre_filter=True  stage1=True  judge=True
RAG:       123 прикладів з data/rag/chroma


In [5]:
# ── КЛАСИФІКАЦІЯ (multithreaded + tqdm) ──────────────────────────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import threading

CONCURRENCY = 20  # кількість потоків

found_counter = {"n": 0}
found_lock = threading.Lock()

def _classify_row(row_dict):
    res = classify_provider_message(
        text       = row_dict['text_merged'],
        glossary   = glossary,
        llm        = llm,
        cfg        = cfg,
        rag_index  = provider_rag,
        message_id = row_dict.get('message_id'),
        username   = row_dict.get('username'),
    )

    if isinstance(res, list):
        res = res[0] if len(res) > 0 else {}

    res['date']          = row_dict.get('date', '')
    res['group_id']      = row_dict.get('group_id', '')
    res['text_original'] = row_dict.get('text', '')
    res['_idx']          = row_dict['_idx']

    is_confirmed = (
        res.get('is_provider') and
        res.get('verdict') != 'MISTAKE'
    )
    if is_confirmed:
        with found_lock:
            found_counter["n"] += 1

    return res

# Готуємо список dict-ів для потоків
rows = []
for i, (_, row) in enumerate(df.iterrows()):
    d = row.to_dict()
    d['_idx'] = i
    rows.append(d)

total = len(rows)
results = []

with ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
    futures = {pool.submit(_classify_row, r): r for r in rows}
    with tqdm(total=total, desc="Класифікація", unit="msg", mininterval=5) as pbar:
        for fut in as_completed(futures):
            try:
                res = fut.result()
                results.append(res)
            except Exception as exc:
                print(exc)
                results.append({'_idx': futures[fut]['_idx'], 'error': str(exc)})
            pbar.set_postfix(providers=found_counter["n"])
            pbar.update(1)

# Відновлюємо оригінальний порядок
results.sort(key=lambda r: r.get('_idx', 0))
for r in results:
    r.pop('_idx', None)

results_df = pd.DataFrame(results)
found = found_counter["n"]

usage = llm.usage.to_dict()
print(f'\nКласифікацію завершено. Підтверджених провайдерів: {found}')
print(f'LLM викликів:          {usage["calls"]:,}')
print(f'Токенів (in + out):    {usage["input_tokens"]:,} + {usage["output_tokens"]:,}')
print(f'Вартість (est.):       ${usage["estimated_cost_usd"]:.4f}')
print(f'Вартість / повідомлення: ${usage["estimated_cost_usd"] / max(total, 1):.6f}')
if usage.get('by_stage'):
    print('\nПо стадіях:')
    for stage, s in usage['by_stage'].items():
        print(f'  {stage:<35} calls={s["calls"]:>5}  '
              f'tokens={s["input_tokens"]+s["output_tokens"]:>8,}  '
              f'${s["estimated_cost_usd"]:.4f}')

Класифікація:   1%|          | 200/19361 [00:18<31:11, 10.24msg/s, providers=10]

'list' object has no attribute 'get'


Класифікація:   1%|▏         | 267/19361 [00:22<28:39, 11.11msg/s, providers=13]

'list' object has no attribute 'get'


Класифікація:   9%|▊         | 1678/19361 [02:11<21:42, 13.57msg/s, providers=110]

...

'list' object has no attribute 'get'


Класифікація:  90%|█████████ | 17479/19361 [21:31<02:31, 12.43msg/s, providers=927]

'list' object has no attribute 'get'


Класифікація: 100%|██████████| 19361/19361 [24:14<00:00, 13.31msg/s, providers=1050]


Класифікацію завершено. Підтверджених провайдерів: 1050
LLM викликів:          19,289
Токенів (in + out):    33,598,877 + 1,128,169
Вартість (est.):       $0.0000
Вартість / повідомлення: $0.000000

По стадіях:
  provider_stage1_micro               calls= 8250  tokens=14,723,135  $0.0000
  provider_full_prompt                calls= 7295  tokens=13,287,221  $0.0000
  provider_judge                      calls= 3744  tokens=6,716,690  $0.0000


In [6]:
# ── ПІДСУМКИ ─────────────────────────────────────────────────────────────────
from collections import Counter

total_msg  = len(results_df)

# Провайдери: is_provider=True І verdict != MISTAKE
confirmed  = results_df[
    results_df['is_provider'].astype(bool) &
    (results_df['verdict'] != 'MISTAKE')
]
# Відхилені суддею
judged_out = results_df[
    results_df['is_provider'].astype(bool) &
    (results_df['verdict'] == 'MISTAKE')
]
# На review
on_review  = results_df[
    results_df['is_provider'].astype(bool) &
    (results_df['verdict'] == 'REVIEW')
]

print('=' * 55)
print('  РЕЗУЛЬТАТИ')
print('=' * 55)
print(f'  Повідомлень прогнано        : {total_msg:,}')
print(f'  Підтверджені провайдери     : {len(confirmed):,}  ({len(confirmed)/total_msg*100:.1f}%)')
print(f'  Відхилені суддею (MISTAKE)  : {len(judged_out):,}')
print(f'  На перевірку (REVIEW)       : {len(on_review):,}')

# Статистика stage breakdown
pre_filtered  = (results_df['rationale'].str.contains('pre-filtered', na=False)).sum()
stage1_filtered = (results_df['rationale'].str.contains('stage1-filtered', na=False)).sum()
print(f'\n  Відсіяно pre-filter         : {pre_filtered:,}')
print(f'  Відсіяно stage1 micro-LLM   : {stage1_filtered:,}')

if len(confirmed):
    print(f'\n  Впевненість (avg)           : {confirmed["confidence"].mean():.2f}')
    print(f'  Впевненість (min)           : {confirmed["confidence"].min():.2f}')

    # Топ вертикалей
    verticals = Counter()
    for v in confirmed['vertical']:
        for item in str(v).split(','):
            item = item.strip()
            if item and item not in ('', 'nan', 'unknown'):
                verticals[item] += 1
    if verticals:
        print(f'\n  Топ вертикалей:')
        for v, cnt in verticals.most_common(5):
            print(f'    {v:<25} {cnt}')

    # Топ GEO
    geos = Counter()
    for g in confirmed['geo']:
        for item in str(g).split(','):
            item = item.strip()
            if item and item not in ('', 'nan'):
                geos[item] += 1
    if geos:
        print(f'\n  Топ GEO:')
        for g, cnt in geos.most_common(5):
            print(f'    {g:<25} {cnt}')

# Вартість
print()
print('=' * 55)
print('  ВАРТІСТЬ')
print('=' * 55)
usage = llm.usage.to_dict()
print(f'  LLM викликів               : {usage["calls"]:,}')
print(f'  Токенів (in + out)         : {usage["input_tokens"]:,} + {usage["output_tokens"]:,}')
print(f'  Вартість (est.)            : ${usage["estimated_cost_usd"]:.4f}')
print(f'  Вартість / повідомлення    : ${usage["estimated_cost_usd"] / max(total_msg, 1):.6f}')
print()
if usage.get('by_stage'):
    print('  По стадіях:')
    for stage, s in usage['by_stage'].items():
        print(f'    {stage:<35} calls={s["calls"]:>4}  '
              f'tokens={s["input_tokens"]+s["output_tokens"]:>7,}  '
              f'${s["estimated_cost_usd"]:.4f}')

  РЕЗУЛЬТАТИ
  Повідомлень прогнано        : 19,361
  Підтверджені провайдери     : 1,072  (5.5%)
  Відхилені суддею (MISTAKE)  : 2,719
  На перевірку (REVIEW)       : 0

  Відсіяно pre-filter         : 11,227
  Відсіяно stage1 micro-LLM   : 748

  Впевненість (avg)           : 0.92
  Впевненість (min)           : 0.80

  Топ вертикалей:
    other                     553
    igaming                   209
    casino                    173
    forex                     145
    sportsbook                124

  Топ GEO:
    RU                        160
    EU                        100
    LATAM                     79
    Asia                      59
    China                     48

  ВАРТІСТЬ
  LLM викликів               : 19,289
  Токенів (in + out)         : 33,598,877 + 1,128,169
  Вартість (est.)            : $0.0000
  Вартість / повідомлення    : $0.000000

  По стадіях:
    provider_stage1_micro               calls=8250  tokens=14,723,135  $0.0000
    provider_full_prompt         

In [7]:
# ── ЕКСПОРТ ───────────────────────────────────────────────────────────────────
# Лист 1: REAL_PROVIDER (підтверджені)
# Лист 2: REVIEW (на перевірку)
# Лист 3: всі результати
# Лист 4: summary + вартість

COLS_PROVIDERS = [
    'message_id', 'username', 'date',
    'verdict', 'confidence',
    'company', 'geo', 'methods', 'vertical',
    'evidence_quote', 'rationale', 'judge_reason',
    'glossary_terms_seen', 'text',
]
COLS_ALL = COLS_PROVIDERS + ['is_provider', 'elapsed_ms', 'text_original']

WIDTHS = {
    'message_id': 12, 'username': 18, 'date': 18,
    'verdict': 14, 'confidence': 12,
    'company': 25, 'geo': 20, 'methods': 25, 'vertical': 20,
    'evidence_quote': 50, 'rationale': 55, 'judge_reason': 45,
    'glossary_terms_seen': 40, 'text_merged': 80,
    'is_provider': 12, 'elapsed_ms': 12, 'text_original': 60,
}

GREEN  = PatternFill('solid', fgColor='D6F0D6')  # REAL_PROVIDER
YELLOW = PatternFill('solid', fgColor='FFF7CC')  # REVIEW
RED    = PatternFill('solid', fgColor='F5D0CE')  # MISTAKE / not provider
GRAY   = PatternFill('solid', fgColor='F2F2F2')  # not provider (pre/stage1 filtered)
HDR    = PatternFill('solid', fgColor='1A3A5C')
THIN   = Border(
    left=Side(style='thin', color='CCCCCC'), right=Side(style='thin', color='CCCCCC'),
    top=Side(style='thin', color='CCCCCC'),  bottom=Side(style='thin', color='CCCCCC'),
)

def row_fill(r):
    if r.get('verdict') == 'REAL_PROVIDER':
        return GREEN
    if r.get('verdict') == 'REVIEW':
        return YELLOW
    if r.get('verdict') == 'MISTAKE':
        return RED
    return GRAY

def write_sheet(ws, data, cols):
    actual = [c for c in cols if c in data.columns]
    for ci, col in enumerate(actual, 1):
        cell = ws.cell(row=1, column=ci, value=col)
        cell.fill = HDR
        cell.font = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center')
        cell.border = THIN
        ws.column_dimensions[cell.column_letter].width = WIDTHS.get(col, 15)
    for ri, (_, row) in enumerate(data[actual].iterrows(), 2):
        fill = row_fill(row)
        for ci, col in enumerate(actual, 1):
            val  = row[col]
            cell = ws.cell(row=ri, column=ci, value=str(val) if val is not None else '')
            cell.fill   = fill
            cell.border = THIN
            cell.alignment = Alignment(
                wrap_text=(col in ('text', 'text_original', 'rationale',
                                   'evidence_quote', 'judge_reason')),
                vertical='top',
            )
        ws.row_dimensions[ri].height = 55
    ws.freeze_panes = 'A2'
    if len(data):
        ws.auto_filter.ref = f"A1:{ws.cell(1, len(actual)).column_letter}1"

wb = openpyxl.Workbook()

# ── ЕКСПОРТ ───────────────────────────────────────────────────────────────────
# Лист 1: REAL_PROVIDER (підтверджені)
# Лист 2: REAL_PROVIDER (дедупліковані по text)
# Лист 3: REVIEW (на перевірку)
# Лист 4: всі результати
# Лист 5: summary + вартість

# 1. Создаем дедуплицированный набор только для подтвержденных провайдеров
confirmed_dedup = confirmed.drop_duplicates(subset=['text'], keep='first')

wb = openpyxl.Workbook()

# Лист 1 — REAL_PROVIDER
ws1 = wb.active
ws1.title = 'real_providers'
write_sheet(ws1, confirmed, COLS_PROVIDERS)

# Лист 2 — REAL_PROVIDER (без дубликатов по text)
ws2 = wb.create_sheet('real_providers_dedup')
write_sheet(ws2, confirmed_dedup, COLS_PROVIDERS)

# Лист 3 — REVIEW
ws3 = wb.create_sheet('review')
write_sheet(ws3, on_review, COLS_PROVIDERS)

# Лист 4 — всі результати
ws4 = wb.create_sheet('all_results')
write_sheet(ws4, results_df, COLS_ALL)

# Лист 5 — summary
ws5 = wb.create_sheet('summary')
summary_rows = [
    ('Метрика',                         'Значення'),
    ('Повідомлень прогнано',            total_msg),
    ('Підтверджені провайдери',         len(confirmed)),
    ('Підтверджені (дедуп по text)',   len(confirmed_dedup)),  # <--- Добавили метрику
    ('Відхилені суддею (MISTAKE)',      len(judged_out)),
    ('На перевірку (REVIEW)',           len(on_review)),
    ('Відсіяно pre-filter',            pre_filtered),
    ('Відсіяно stage1 micro-LLM',      stage1_filtered),
    ('', ''),
    ('LLM викликів',                   usage['calls']),
    ('Токенів вхідних',                usage['input_tokens']),
    ('Токенів вихідних',               usage['output_tokens']),
    ('Вартість (est. USD)',            f'${usage["estimated_cost_usd"]:.4f}'),
    ('Вартість / повідомлення',        f'${usage["estimated_cost_usd"] / max(total_msg, 1):.6f}'),
    ('', ''),
    ('Модель',                         llm.model),
    ('Глосарій термінів',              len(glossary)),
    ('pre_filter',                     str(PRE_FILTER_ENABLED)),
    ('stage1',                         str(STAGE1_ENABLED)),
    ('judge',                          str(JUDGE_ENABLED)),
]
if usage.get('by_stage'):
    summary_rows.append(('', ''))
    summary_rows.append(('По стадіях:', ''))
    for stage, s in usage['by_stage'].items():
        summary_rows.append((
            stage,
            f'calls={s["calls"]}  tokens={s["input_tokens"]+s["output_tokens"]:,}  ${s["estimated_cost_usd"]:.4f}'
        ))

for r, (a, b) in enumerate(summary_rows, 1):
    ca = ws5.cell(row=r, column=1, value=a)
    cb = ws5.cell(row=r, column=2, value=b)
    ca.font = Font(bold=(r == 1))
ws5.column_dimensions['A'].width = 35
ws5.column_dimensions['B'].width = 45

wb.save(OUTPUT_FILE)
print(f'Збережено → {OUTPUT_FILE}')
print(f'  Лист "real_providers"       : {len(confirmed)} провайдерів  (зелений)')
print(f'  Лист "real_providers_dedup" : {len(confirmed_dedup)} унікальних по text')
print(f'  Лист "review"               : {len(on_review)} на перевірку  (жовтий)')
print(f'  Лист "all_results"          : {total_msg} всього')
print(f'  Лист "summary"              : вартість + конфіг')

IllegalCharacterError: Даем работу в круг Страховой гу/мобком 300$
Другие методы 500$
От меня - выдача актуальных мануалов, полное курирование, обучение. cannot be used in worksheets.